# cross-product-normal — ex2: batched unit normals + degenerate-triangle mask

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `cross-product-normal`. Running the final beacon cell reports progress against the `Geometry: Cross-product surface normal` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Geometry: Cross-product surface normal` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`cross-product-normal`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "cross-product-normal"
DD_SUBTOPIC = "Geometry: Cross-product surface normal"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Cross-product surface normal — deepening refresher

For a single triangle, `n = cross(P2-P1, P3-P1)` then normalize. For a BATCH of `(N, 3, 3)` triangles (N triangles, each with 3 vertices, each vertex 3-D), the same recipe applies along the last axis:

```python
e1 = tris[:, 1] - tris[:, 0]          # (N, 3)
e2 = tris[:, 2] - tris[:, 0]          # (N, 3)
n = t.linalg.cross(e1, e2, dim=-1)     # (N, 3) — un-normalized
norms = n.norm(dim=-1, keepdim=True)   # (N, 1)
```

**Degeneracy.** A triangle is degenerate iff its three vertices are colinear ⇒ `cross == 0` ⇒ `||n|| == 0` ⇒ division by zero produces `nan` / `inf`. Real renderers test `norms > eps` (eps ~ 1e-8) and either skip the triangle or substitute a sentinel normal.

**Returning a mask is better than skipping.** A boolean `valid: (N,)` lets downstream code decide what to do (skip in lighting, but maybe keep for connectivity).

### Exercise 2 — batched unit normals + degenerate-triangle mask

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `t.linalg.cross(..., dim=-1)` over a `(N, 3, 3)` batch of triangle vertices and produce both the unit-normals tensor AND a boolean mask of degenerate (colinear) triangles.
> Keywords: cross-product, batched, degenerate, mask
> ```

**KCs targeted:** `batched-cross-along-dim-minus-1`, `degenerate-norm-mask`

Implement `ex2_batched_normals(tris, eps=1e-8)`. Compute unit surface normals for a batch of triangles AND flag the degenerate ones.

1. `tris` has shape `(N, 3, 3)` — N triangles, 3 vertices each, 3-D coords.
2. Edges from shared vertex `P1 = tris[:, 0]`:
   ```python
   e1 = tris[:, 1] - tris[:, 0]   # (N, 3)
   e2 = tris[:, 2] - tris[:, 0]   # (N, 3)
   ```
3. Cross product along the LAST axis: `n = t.linalg.cross(e1, e2, dim=-1)`. Shape `(N, 3)`.
4. Norms: `norms = n.norm(dim=-1, keepdim=True)` — shape `(N, 1)`.
5. Degenerate mask: `valid = (norms.squeeze(-1) > eps)` — shape `(N,)`, `True` iff the triangle has nonzero area.
6. Normalize SAFELY. For degenerate triangles, division would produce `nan` / `inf`; replace the denominator with `1.0` where the triangle is invalid (the resulting normal there is the zero vector — a sentinel that downstream code can detect).
   ```python
   safe_norms = norms.clamp(min=eps)
   unit = n / safe_norms
   # zero-out the degenerate ones so the value is a clean sentinel
   unit[~valid] = 0.0
   ```
7. Return `(unit, valid)`. `unit` has shape `(N, 3)`; `valid` has shape `(N,)` (bool).

**Do NOT** call `ex1_triangle_normal` in a Python loop. Use batched ops throughout.

Input: `tris` shape `(N, 3, 3)` float; `eps` float.
Output: tuple `(unit_normals (N,3), valid_mask (N,))`.

In [ ]:
def ex2_batched_normals(tris: Tensor, eps: float = 1e-8) -> tuple:
    """Return (unit_normals (N,3), valid_mask (N,)) for a batch of triangles."""
    raise NotImplementedError()


def _test_ex2():
    import math

    # Mixed batch: 3 valid + 1 degenerate (colinear).
    tris = t.tensor([
        # CCW in z=0 plane → +z normal
        [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [0.0, 1.0, 0.0]],
        # x=5 plane → +x normal
        [[5.0, 0.0, 0.0], [5.0, 1.0, 0.0], [5.0, 0.0, 1.0]],
        # tilted triangle
        [[0.0, 0.0, 0.0], [2.0, 1.0, 0.0], [0.0, 1.0, 3.0]],
        # DEGENERATE: three colinear points along x-axis
        [[0.0, 0.0, 0.0], [1.0, 0.0, 0.0], [2.0, 0.0, 0.0]],
    ])
    unit, valid = ex2_batched_normals(tris)
    assert unit.shape == (4, 3), f'expected (4, 3), got {tuple(unit.shape)}'
    assert valid.shape == (4,), f'expected (4,), got {tuple(valid.shape)}'
    assert valid.dtype == t.bool, f'mask must be bool, got {valid.dtype}'

    # Valid mask: first 3 True, last False.
    expected_valid = t.tensor([True, True, True, False])
    assert t.equal(valid, expected_valid), f'valid mask wrong: {valid}'

    # Triangle 0: +z.
    assert t.allclose(unit[0], t.tensor([0.0, 0.0, 1.0]), atol=1e-5), f'tri 0 normal: {unit[0]}'
    # Triangle 1: +x.
    assert t.allclose(unit[1], t.tensor([1.0, 0.0, 0.0]), atol=1e-5), f'tri 1 normal: {unit[1]}'
    # Triangle 2: must be unit length and perpendicular to both edges.
    n2 = unit[2]
    assert abs(n2.norm().item() - 1.0) < 1e-5, f'tri 2 not unit: |n|={n2.norm().item()}'
    e1_2 = tris[2, 1] - tris[2, 0]
    e2_2 = tris[2, 2] - tris[2, 0]
    assert abs((n2 * e1_2).sum().item()) < 1e-5, f'tri 2 not perp to e1: dot={(n2*e1_2).sum().item()}'
    assert abs((n2 * e2_2).sum().item()) < 1e-5, f'tri 2 not perp to e2: dot={(n2*e2_2).sum().item()}'
    # Triangle 3: degenerate → zero-vector sentinel.
    assert t.allclose(unit[3], t.zeros(3), atol=1e-7), (
        f'degenerate triangle must have zero-vector normal (sentinel), got {unit[3]}'
    )
    # No nan / inf anywhere — that's the whole point of the safe-clamp pattern.
    assert t.isfinite(unit).all(), 'normals must be finite; check the safe-divide pattern'

    # Stress test: 50 random valid triangles + 5 degenerate.
    rng = t.Generator().manual_seed(0)
    rand_valid = t.randn(50, 3, 3, generator=rng)
    # Construct 5 colinear ones explicitly.
    ts_col = []
    for k in range(5):
        a = t.randn(3, generator=rng)
        b = t.randn(3, generator=rng)
        ts_col.append(t.stack([a, a + b, a + 2 * b]))   # all on line through a
    rand_col = t.stack(ts_col)
    big = t.cat([rand_valid, rand_col], dim=0)
    # Use a slightly larger eps to catch the residual float32 error in the
    # colinear construction (cross of nearly-parallel edges leaves ~1e-7 noise).
    unit_b, valid_b = ex2_batched_normals(big, eps=1e-5)
    assert unit_b.shape == (55, 3)
    assert valid_b.shape == (55,)
    assert valid_b[:50].all(), 'random triangles should be valid (probability 1)'
    assert (~valid_b[50:]).all(), 'all 5 colinear triangles must be flagged degenerate'
    # Unit-length for the valid ones.
    valid_norms = unit_b[valid_b].norm(dim=-1)
    assert t.allclose(valid_norms, t.ones_like(valid_norms), atol=1e-5), (
        f'valid normals must be unit length, got norms ranging {valid_norms.min().item():.4f}..{valid_norms.max().item():.4f}'
    )
    # Zero for the degenerate ones.
    assert t.allclose(unit_b[~valid_b], t.zeros_like(unit_b[~valid_b]), atol=1e-7)
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_batched_normals(tris, eps=1e-8):
    e1 = tris[:, 1] - tris[:, 0]                   # (N, 3)
    e2 = tris[:, 2] - tris[:, 0]                   # (N, 3)
    n = t.linalg.cross(e1, e2, dim=-1)             # (N, 3)
    norms = n.norm(dim=-1, keepdim=True)           # (N, 1)
    valid = norms.squeeze(-1) > eps                # (N,) bool
    safe = norms.clamp(min=eps)
    unit = n / safe
    unit[~valid] = 0.0                             # sentinel
    return unit, valid
```

**Why `dim=-1` instead of `dim=1`.** Either works for a `(N, 3)` tensor. `dim=-1` is the convention that survives rank changes — the same code works if you later wrap everything in another batch dim (`(M, N, 3, 3)`).

**Why `clamp(min=eps)` then mask, not just `if-else`.** Branching per-element would require a Python loop or a `torch.where` chain. Clamping the denominator first ensures the division never produces `nan`/`inf`; then a single bool indexer (`unit[~valid] = 0.0`) replaces the bogus values with a clean sentinel. Vectorized + numerically safe.

**Why a zero vector as sentinel.** Downstream code can detect 'this normal is invalid' with `(unit == 0).all(-1)`, which is cheap to vectorize. A unit-length sentinel (e.g. +z) would confuse downstream lighting that takes `dot(L, n)` — a real, non-degenerate +z triangle would look identical to a degenerate one. The zero vector has no valid interpretation, so it's safely distinguishable.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()